# Chess Engine with PyTorch

In [ ]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 86.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=150a93ec4fc1926d13679dd18332e01423408942506ad22bb0d3bb438b981336
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


## Imports

In [2]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 84.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=be78efe3d4586079eddd3b1108250a6733a04761b989c224964b003d2b83e30d
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [3]:
import os
import numpy as np # type: ignore
import time
import torch
import torch.nn as nn # type: ignore
import torch.optim as optim # type: ignore
from torch.utils.data import DataLoader # type: ignore
from chess import pgn # type: ignore
from tqdm import tqdm # type: ignore

# Data preprocessing

## Load data

In [6]:
import os
import gc
import numpy as np
from chess import pgn
from tqdm import tqdm
from auxiliary_func import create_input_for_nn

def load_and_process_pgn_chunked(data_dir, max_samples=1000000, batch_size=2000):
    pgn_files = [f for f in os.listdir(data_dir) if f.endswith(".pgn")]

    all_X = []
    all_y = []
    total_samples = 0

    print(f"Tìm thấy {len(pgn_files)} file PGN. Bắt đầu xử lý cuốn chiếu...")

    for file_name in pgn_files:
        file_path = os.path.join(data_dir, file_name)

        with open(file_path, 'r') as f:
            batch_games = []
            while True:
                try:
                    game = pgn.read_game(f)
                except Exception:
                    continue

                if game is None:
                    break

                batch_games.append(game)

                if len(batch_games) >= batch_size:

                    X_chunk, y_chunk = create_input_for_nn(batch_games)

                    all_X.append(X_chunk)
                    all_y.append(y_chunk)
                    total_samples += len(y_chunk)

                    del batch_games
                    batch_games = []
                    gc.collect()

                    print(f"-> Đã xử lý: {total_samples} nước đi...", end='\r')

                    if max_samples and total_samples >= max_samples:
                        print(f"\nĐã đạt giới hạn {max_samples} mẫu. Dừng đọc.")
                        return np.concatenate(all_X), np.concatenate(all_y)

            if batch_games:
                X_chunk, y_chunk = create_input_for_nn(batch_games)
                all_X.append(X_chunk)
                all_y.append(y_chunk)
                total_samples += len(y_chunk)
                del batch_games
                gc.collect()

        if max_samples and total_samples >= max_samples:
             break

    if total_samples == 0:
        print("Không tìm thấy dữ liệu hoặc lỗi đọc file.")
        return np.array([]), np.array([])

    print("\nĐang gộp các mảnh dữ liệu (Concatenating)...")
    final_X = np.concatenate(all_X, axis=0)
    final_y = np.concatenate(all_y, axis=0)

    if max_samples and len(final_y) > max_samples:
        final_X = final_X[:max_samples]
        final_y = final_y[:max_samples]

    return final_X, final_y

MAX_SAMPLES = 1000000
X, y = load_and_process_pgn_chunked("/content/data/pgn", max_samples=MAX_SAMPLES)

print(f"\nHOÀN TẤT! Số mẫu dữ liệu cuối cùng: {len(y)}")

Tìm thấy 1 file PGN. Bắt đầu xử lý cuốn chiếu...
-> Đã xử lý: 1159575 nước đi...
Đã đạt giới hạn 1000000 mẫu. Dừng đọc.

HOÀN TẤT! Số mẫu dữ liệu cuối cùng: 1159575


## Convert data into tensors

In [8]:
from auxiliary_func import create_input_for_nn, encode_moves

In [9]:
y, move_to_int = encode_moves(y)
num_classes = len(move_to_int)

In [10]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

# Preliminary actions

In [11]:
from dataset import ChessDataset
from model import ChessModel

In [12]:
# Create Dataset and DataLoader
dataset = ChessDataset(X, y)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

# Model Initialization
model = ChessModel(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

Using device: cuda


# Training

In [13]:
num_epochs = 50
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
        optimizer.zero_grad()

        outputs = model(inputs)  # Raw logits

        # Compute loss
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        running_loss += loss.item()
    end_time = time.time()
    epoch_time = end_time - start_time
    minutes: int = int(epoch_time // 60)
    seconds: int = int(epoch_time) - minutes * 60
    print(f'Epoch {epoch + 1 + 50}/{num_epochs + 1 + 50}, Loss: {running_loss / len(dataloader):.4f}, Time: {minutes}m{seconds}s')

100%|██████████| 18119/18119 [01:07<00:00, 270.31it/s]


Epoch 51/101, Loss: 4.1385, Time: 1m7s


100%|██████████| 18119/18119 [01:06<00:00, 274.42it/s]


Epoch 52/101, Loss: 3.0817, Time: 1m6s


100%|██████████| 18119/18119 [01:05<00:00, 278.67it/s]


...

Epoch 98/101, Loss: 1.0843, Time: 1m3s


100%|██████████| 18119/18119 [01:03<00:00, 284.32it/s]


Epoch 99/101, Loss: 1.0724, Time: 1m3s


100%|██████████| 18119/18119 [01:04<00:00, 280.99it/s]

Epoch 100/101, Loss: 1.0611, Time: 1m4s


# Save the model and mapping

In [14]:
# Save the model
torch.save(model.state_dict(), "/content/TORCH_50EPOCHS.pth")

In [16]:
import pickle

with open("/content/move_to_int_50EPOCHS", "wb") as file:
    pickle.dump(move_to_int, file)